In [0]:
CREATE TABLE IF NOT EXISTS ecommerce_lakehouse.gold.dim_customer
(
    customer_sk BIGINT,
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    effective_start_date DATE,
    effective_end_date DATE,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
%python
from pyspark.sql.functions import (
    monotonically_increasing_id,
    current_date,
    lit
)
customers = spark.table(
    "ecommerce_lakehouse.silver.customers"
)
initial_customer_dim = (
    customers
    .select(
        monotonically_increasing_id()
            .alias("customer_sk"),

        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
    .withColumn(
        "effective_start_date",
        current_date()
    )
    .withColumn(
        "effective_end_date",
        lit("9999-12-31").cast("date")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)
(
    initial_customer_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ecommerce_lakehouse.gold.dim_customer"
    )
)

In [0]:
%python
from delta.tables import DeltaTable

customer_dimension = DeltaTable.forName(
    spark,
    "ecommerce_lakehouse.gold.dim_customer"
)

customer_changes = (
    spark.table(
        "ecommerce_lakehouse.silver.customers"
    )
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
)

(
    customer_dimension.alias("target")
    .merge(
        customer_changes.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="""
        target.customer_city <> source.customer_city
        OR target.customer_state <> source.customer_state
        OR target.customer_zip_code_prefix <> source.customer_zip_code_prefix
        """,
        set={
            "effective_end_date": "current_date()",
            "is_current": "false"
        }
    )
    .execute()
)

(
    customer_dimension.alias("target")
    .merge(
        customer_changes.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        """
    )
    .whenNotMatchedInsert(
        values={
            "customer_sk": "monotonically_increasing_id()",
            "customer_id": "source.customer_id",
            "customer_unique_id": "source.customer_unique_id",
            "customer_zip_code_prefix": "source.customer_zip_code_prefix",
            "customer_city": "source.customer_city",
            "customer_state": "source.customer_state",
            "effective_start_date": "current_date()",
            "effective_end_date": "CAST('9999-12-31' AS DATE)",
            "is_current": "true"
        }
    )
    .execute()
)



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]